# Human Pose Estimation using Heatmaps (MMPose)

Research-internship project: **heatmap-based 2-D human pose estimation** with
[MMPose](https://github.com/open-mmlab/mmpose), analysis of the
**joint-confidence heatmaps** with OpenCV + Matplotlib, and **validation against
Kinovea** motion-analysis measurements.

| Step | What happens |
|---|---|
| 1 | Install MMPose (MMEngine + MMCV) on Colab |
| 2 | Write the project code (`posestudy/` package) |
| 3 | Run HRNet-W32 (top-down heatmap model, COCO-17 joints) on an image / video, keeping the raw heatmaps |
| 4 | Visualise heatmaps, measure how well each joint is localised, flag unreliable predictions |
| 5 | Compare keypoints and joint angles with Kinovea (error px, PCK, angle MAE, Bland-Altman) |

**Runtime → Change runtime type → GPU (T4)** before starting.

## 1. Setup

In [ ]:
!nvidia-smi -L || echo "No GPU - inference will run on CPU (slow but works)"
import torch, sys
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| python", sys.version.split()[0])

In [ ]:
#@title Optional: mount Google Drive (keeps results + the compiled MMCV wheel between sessions)
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/pose-heatmap-study"  #@param {type:"string"}
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    import os; os.makedirs(DRIVE_DIR, exist_ok=True)

### Install MMPose

MMPose 1.x needs **MMCV with compiled CUDA ops**. OpenMMLab has no prebuilt MMCV
wheels for the PyTorch version Colab ships today, so MMCV is compiled from source
the first time (**roughly 30-60 min on Colab** — a CPU-only build took 23 min on 4 cores; Colab has 2 cores and also compiles CUDA kernels). The wheel is then saved to Drive, and
later sessions install it in seconds (as long as Colab's PyTorch version stays
the same).

Other notes:
* `mmpose` is installed with `--no-deps` to skip `chumpy` (only needed for SMPL, fails to build on Python 3.12).
* MMPose imports a few `mmdet` utilities, so `mmdet` is installed (`--no-deps`) and its MMCV version check is relaxed. People are detected with torchvision's Faster R-CNN, not mmdet.
* `setuptools<70` is needed because MMCV's `setup.py` still imports `pkg_resources`.

In [ ]:
import glob, os, subprocess, sys, torch

def mmcv_ok():
    r = subprocess.run([sys.executable, "-c", "from mmcv.ops import nms; import mmcv; print(mmcv.__version__)"],
                       capture_output=True, text=True)
    return r.returncode == 0

!pip install -q "setuptools<70" ninja "mmengine==0.10.7" json_tricks munkres
# --no-cache-dir: compile xtcocotools against the NumPy that is installed now
# (a cached wheel built for another NumPy fails with "numpy.dtype size changed")
!pip install -q --no-cache-dir --no-build-isolation xtcocotools

tag = f"torch{torch.__version__.split('+')[0]}_cu{(torch.version.cuda or 'cpu').replace('.', '')}"
wheel_dir = f"{DRIVE_DIR}/wheels/{tag}" if USE_DRIVE else f"/content/wheels/{tag}"
os.makedirs(wheel_dir, exist_ok=True)

if not mmcv_ok():
    cached = glob.glob(f"{wheel_dir}/mmcv-2.2.0*.whl")
    if cached:
        print("Installing cached MMCV wheel:", cached[0])
        !pip install -q {cached[0]}
    else:
        print(f"Compiling MMCV 2.2.0 for {tag} - this takes a while (only once) ...")
        os.environ.update(MMCV_WITH_OPS="1", FORCE_CUDA="1" if torch.cuda.is_available() else "0",
                          MAX_JOBS=str(os.cpu_count()))
        !pip wheel --no-build-isolation --no-deps "mmcv==2.2.0" -w {wheel_dir}
        built = glob.glob(wheel_dir + "/mmcv-2.2.0*.whl")[0]
        !pip install -q {built}

!pip install -q --no-deps "mmpose==1.3.2" "mmdet==3.3.0"
# mmpose imports a few mmdet utilities; mmdet asserts mmcv < 2.2.0 but works fine with 2.2.0
import importlib.util, re
init = importlib.util.find_spec("mmdet").origin
src = open(init).read()
open(init, "w").write(re.sub(r"mmcv_maximum_version = '[0-9.]+'", "mmcv_maximum_version = '2.3.0'", src))
assert mmcv_ok(), "MMCV ops failed to import - see the build log above"
!python -c "import mmcv, mmengine, mmpose; print('mmcv', mmcv.__version__, '| mmengine', mmengine.__version__, '| mmpose', mmpose.__version__)"

## 2. Project code

These cells write the `posestudy` package and command-line scripts to
`/content/pose-heatmap-study`. Same code as the GitHub repo.

In [ ]:
import os
os.makedirs("/content/pose-heatmap-study/posestudy", exist_ok=True)
os.makedirs("/content/pose-heatmap-study/scripts", exist_ok=True)
os.makedirs("/content/pose-heatmap-study/data", exist_ok=True)
%cd /content/pose-heatmap-study

In [ ]:
%%writefile posestudy/__init__.py
"""Human pose estimation with heatmap-based models (MMPose).

Modules
-------
skeleton   : COCO keypoint names, skeleton edges, joint-angle helpers (no MMPose needed)
estimator  : MMPose model loading + per-frame inference that keeps the raw heatmaps
heatmaps   : heatmap quality metrics and OpenCV / Matplotlib visualisations
kinovea    : loading Kinovea exports and validating predictions against them
"""

__version__ = "1.0.0"

In [ ]:
%%writefile posestudy/skeleton.py
"""COCO-17 keypoint definitions and joint-angle helpers.

Only depends on NumPy, so it can be used for analysis without MMPose installed.
"""

import numpy as np

COCO_KEYPOINTS = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle",
]
KPT_INDEX = {name: i for i, name in enumerate(COCO_KEYPOINTS)}

# Pairs of keypoint names joined by a bone when drawing the skeleton.
SKELETON = [
    ("left_ankle", "left_knee"), ("left_knee", "left_hip"),
    ("right_ankle", "right_knee"), ("right_knee", "right_hip"),
    ("left_hip", "right_hip"), ("left_shoulder", "left_hip"),
    ("right_shoulder", "right_hip"), ("left_shoulder", "right_shoulder"),
    ("left_shoulder", "left_elbow"), ("right_shoulder", "right_elbow"),
    ("left_elbow", "left_wrist"), ("right_elbow", "right_wrist"),
    ("left_eye", "right_eye"), ("nose", "left_eye"), ("nose", "right_eye"),
    ("left_eye", "left_ear"), ("right_eye", "right_ear"),
    ("left_ear", "left_shoulder"), ("right_ear", "right_shoulder"),
]
SKELETON_IDX = [(KPT_INDEX[a], KPT_INDEX[b]) for a, b in SKELETON]

# Joint angle = interior angle at the middle keypoint (degrees, 0-180).
JOINT_ANGLES = {
    "left_elbow": ("left_shoulder", "left_elbow", "left_wrist"),
    "right_elbow": ("right_shoulder", "right_elbow", "right_wrist"),
    "left_shoulder": ("left_elbow", "left_shoulder", "left_hip"),
    "right_shoulder": ("right_elbow", "right_shoulder", "right_hip"),
    "left_hip": ("left_shoulder", "left_hip", "left_knee"),
    "right_hip": ("right_shoulder", "right_hip", "right_knee"),
    "left_knee": ("left_hip", "left_knee", "left_ankle"),
    "right_knee": ("right_hip", "right_knee", "right_ankle"),
}

# BGR colours: left side orange, right side blue, centre green.
def kpt_color(name):
    if name.startswith("left"):
        return (0, 140, 255)
    if name.startswith("right"):
        return (255, 120, 0)
    return (0, 200, 0)


def angle_between(a, b, c):
    """Interior angle ABC in degrees. a, b, c are (..., 2) arrays."""
    a, b, c = (np.asarray(p, dtype=float) for p in (a, b, c))
    v1, v2 = a - b, c - b
    dot = np.sum(v1 * v2, axis=-1)
    norm = np.linalg.norm(v1, axis=-1) * np.linalg.norm(v2, axis=-1)
    with np.errstate(invalid="ignore", divide="ignore"):
        cos = np.clip(dot / norm, -1.0, 1.0)
    return np.degrees(np.arccos(cos))


def compute_joint_angles(keypoints, angle_defs=JOINT_ANGLES):
    """keypoints: (T, 17, 2) -> dict angle_name -> (T,) array of degrees."""
    kp = np.asarray(keypoints, dtype=float)
    return {
        name: angle_between(kp[..., KPT_INDEX[a], :], kp[..., KPT_INDEX[b], :],
                            kp[..., KPT_INDEX[c], :])
        for name, (a, b, c) in angle_defs.items()
    }

In [ ]:
%%writefile posestudy/estimator.py
"""MMPose top-down heatmap inference that keeps the raw per-joint heatmaps.

Pipeline per frame:
    1. find the person box (torchvision Faster R-CNN, or the whole frame)
    2. MMPose top-down heatmap model (HRNet-W32 by default) on that crop
    3. keep keypoints (image px), keypoint scores (= heatmap peak value)
       and the raw 17 x 64 x 48 heatmaps + the crop geometry, so the
       heatmaps can be projected back onto the image later.
"""

import json
import os
import os.path as osp
from glob import glob

import cv2
import numpy as np

from .skeleton import COCO_KEYPOINTS

# name -> (config path inside the mmpose package, checkpoint URL)
_CFG_DIR = "configs/body_2d_keypoint/topdown_heatmap/coco"
_CKPT = "https://download.openmmlab.com/mmpose/v1/body_2d_keypoint/topdown_heatmap/coco/"
MODEL_ZOO = {
    "hrnet-w32": (f"{_CFG_DIR}/td-hm_hrnet-w32_8xb64-210e_coco-256x192.py",
                  _CKPT + "td-hm_hrnet-w32_8xb64-210e_coco-256x192-81c58e40_20220909.pth"),
    "hrnet-w48": (f"{_CFG_DIR}/td-hm_hrnet-w48_8xb32-210e_coco-256x192.py",
                  _CKPT + "td-hm_hrnet-w48_8xb32-210e_coco-256x192-0e67c616_20220913.pth"),
    "resnet50": (f"{_CFG_DIR}/td-hm_res50_8xb64-210e_coco-256x192.py",
                 _CKPT + "td-hm_res50_8xb64-210e_coco-256x192-04af38ce_20220923.pth"),
}


def _default_device():
    import torch
    return "cuda:0" if torch.cuda.is_available() else "cpu"


def mmpose_config_path(rel_path):
    """Resolve a config shipped inside the installed mmpose package."""
    import mmpose
    root = osp.dirname(mmpose.__file__)
    for base in (osp.join(root, ".mim"), osp.dirname(root)):
        path = osp.join(base, rel_path)
        if osp.exists(path):
            return path
    raise FileNotFoundError(f"Could not find {rel_path} inside the mmpose install")


def load_pose_model(name="hrnet-w32", device=None, config=None, checkpoint=None):
    """Build an MMPose top-down heatmap model that also returns its heatmaps.

    `name` picks an entry from MODEL_ZOO; pass `config`/`checkpoint` to use
    any other MMPose top-down heatmap model instead.
    """
    # PyTorch >= 2.6 defaults torch.load(weights_only=True), which rejects the
    # metadata stored in OpenMMLab checkpoints. They are official files, so allow it.
    os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")
    from mmpose.apis import init_model

    if config is None:
        rel_cfg, url = MODEL_ZOO[name]
        config = mmpose_config_path(rel_cfg)
        checkpoint = url if checkpoint is None else checkpoint
    device = device or _default_device()
    # output_heatmaps=True makes MMPose put the heatmaps in pred_fields.
    cfg_options = dict(model=dict(test_cfg=dict(output_heatmaps=True)))
    return init_model(config, checkpoint, device=device, cfg_options=cfg_options)


class PersonDetector:
    """Picks the most confident person box with torchvision Faster R-CNN.

    Keeps the project free of mmdet (whose mmcv version pin clashes with
    current Colab). Good enough for single-subject lab recordings.
    """

    def __init__(self, device=None, score_thr=0.5):
        import torch
        import torchvision
        self.torch = torch
        self.device = device or _default_device()
        weights = torchvision.models.detection.FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
        self.model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(
            weights=weights).eval().to(self.device)
        self.score_thr = score_thr

    def __call__(self, frame_bgr):
        rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        tensor = self.torch.from_numpy(rgb).permute(2, 0, 1).float().div(255)
        with self.torch.no_grad():
            out = self.model([tensor.to(self.device)])[0]
        keep = (out["labels"] == 1) & (out["scores"] >= self.score_thr)  # 1 = person
        if not keep.any():
            return None
        best = out["scores"][keep].argmax()
        return out["boxes"][keep][best].cpu().numpy()


def full_frame_bbox(frame):
    h, w = frame.shape[:2]
    return np.array([0, 0, w, h], dtype=np.float32)


def estimate_frame(model, frame_bgr, bbox):
    """Run the pose model on one person box (x1, y1, x2, y2).

    Returns keypoints (17, 2) in image px, scores (17,), heatmaps (17, h, w)
    in crop space, and the crop centre/scale needed to map them back.
    """
    from mmpose.apis import inference_topdown

    sample = inference_topdown(model, frame_bgr, np.asarray(bbox)[None], bbox_format="xyxy")[0]
    heatmaps = sample.pred_fields.heatmaps
    if hasattr(heatmaps, "cpu"):
        heatmaps = heatmaps.cpu().numpy()
    return dict(
        keypoints=np.asarray(sample.pred_instances.keypoints[0], dtype=np.float32),
        scores=np.asarray(sample.pred_instances.keypoint_scores[0], dtype=np.float32),
        heatmaps=np.asarray(heatmaps, dtype=np.float32),
        input_center=np.asarray(sample.metainfo["input_center"], dtype=np.float32).reshape(2),
        input_scale=np.asarray(sample.metainfo["input_scale"], dtype=np.float32).reshape(2),
    )


def iter_frames(source, stride=1, max_frames=None):
    """Yield (frame_idx, frame_bgr) from a video file, an image, or an image folder."""
    if osp.isdir(source):
        files = sorted(f for ext in ("jpg", "jpeg", "png", "bmp")
                       for f in glob(osp.join(source, f"*.{ext}")))
        frames = ((i, cv2.imread(f)) for i, f in enumerate(files))
    elif source.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
        frames = iter([(0, cv2.imread(source))])
    else:
        def _video():
            cap = cv2.VideoCapture(source)
            if not cap.isOpened():
                raise IOError(f"Cannot open video {source}")
            idx = 0
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                yield idx, frame
                idx += 1
            cap.release()
        frames = _video()

    n = 0
    for idx, frame in frames:
        if idx % stride:
            continue
        yield idx, frame
        n += 1
        if max_frames and n >= max_frames:
            break


def video_fps(source, default=30.0):
    if osp.isdir(source) or source.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
        return default
    cap = cv2.VideoCapture(source)
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    return fps if fps and fps > 0 else default


def run_on_video(model, source, out_dir, detector=None, stride=1, max_frames=None,
                 fps=None, track_bbox=True, model_name="hrnet-w32", verbose=True):
    """Run pose estimation over a whole video and save everything to `out_dir`.

    Saves:
        keypoints.csv   tidy table: frame, time_s, joint, x, y, score
        heatmaps.npz    raw heatmaps (T, 17, h, w) float16 + crop geometry + boxes
        meta.json       fps, frame size, model info
    """
    import pandas as pd

    os.makedirs(out_dir, exist_ok=True)
    fps = fps or video_fps(source)
    rows, frames_idx, kpts, scores, hms, centers, scales, boxes = [], [], [], [], [], [], [], []
    last_bbox, frame_shape = None, None

    for idx, frame in iter_frames(source, stride, max_frames):
        frame_shape = frame.shape[:2]
        bbox = detector(frame) if detector is not None else None
        if bbox is None:
            # No detection: reuse the previous box, else fall back to the full frame.
            bbox = last_bbox if (track_bbox and last_bbox is not None) else full_frame_bbox(frame)
        last_bbox = bbox

        res = estimate_frame(model, frame, bbox)
        frames_idx.append(idx)
        kpts.append(res["keypoints"])
        scores.append(res["scores"])
        hms.append(res["heatmaps"].astype(np.float16))
        centers.append(res["input_center"])
        scales.append(res["input_scale"])
        boxes.append(np.asarray(bbox, dtype=np.float32))
        for j, name in enumerate(COCO_KEYPOINTS):
            x, y = res["keypoints"][j]
            rows.append((idx, idx / fps, name, float(x), float(y), float(res["scores"][j])))
        if verbose and len(frames_idx) % 25 == 0:
            print(f"  processed {len(frames_idx)} frames (frame {idx})")

    if not frames_idx:
        raise RuntimeError(f"No frames read from {source}")

    df = pd.DataFrame(rows, columns=["frame", "time_s", "joint", "x", "y", "score"])
    df.to_csv(osp.join(out_dir, "keypoints.csv"), index=False)
    np.savez_compressed(
        osp.join(out_dir, "heatmaps.npz"),
        frames=np.array(frames_idx), keypoints=np.stack(kpts), scores=np.stack(scores),
        heatmaps=np.stack(hms), input_center=np.stack(centers),
        input_scale=np.stack(scales), bboxes=np.stack(boxes))
    meta = dict(source=osp.abspath(source), fps=fps, height=frame_shape[0],
                width=frame_shape[1], stride=stride, n_frames=len(frames_idx),
                model=model_name, keypoints=COCO_KEYPOINTS)
    with open(osp.join(out_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)
    if verbose:
        print(f"Saved {len(frames_idx)} frames of predictions to {out_dir}")
    return df

In [ ]:
%%writefile posestudy/heatmaps.py
"""Joint-confidence heatmap analysis and visualisation (OpenCV + Matplotlib).

Works on the files written by `estimator.run_on_video`; MMPose is not needed.

For a heatmap model each joint gets a 2-D "probability" map. Its shape tells
us how well the joint was localised:
    peak          max value; MMPose reports it as the keypoint score
    spread_px     spatial std-dev of the map around its centre, in image px
                  (wide blob = uncertain location)
    second_peak   height of the strongest competing peak / main peak
                  (close to 1 = model torn between two places, e.g. left/right swap)
"""

import json
import os
import os.path as osp

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from .skeleton import COCO_KEYPOINTS, SKELETON_IDX, kpt_color


# ----------------------------------------------------------------------------- io
def load_results(out_dir):
    data = dict(np.load(osp.join(out_dir, "heatmaps.npz")))
    data["heatmaps"] = data["heatmaps"].astype(np.float32)
    with open(osp.join(out_dir, "meta.json")) as f:
        data["meta"] = json.load(f)
    return data


def read_frame(source, frame_idx):
    if osp.isdir(source):
        files = sorted(f for f in os.listdir(source)
                       if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")))
        return cv2.imread(osp.join(source, files[frame_idx]))
    if source.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
        return cv2.imread(source)
    cap = cv2.VideoCapture(source)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise IOError(f"Could not read frame {frame_idx} from {source}")
    return frame


# ------------------------------------------------------------------- geometry
def heatmap_to_image(heatmaps, center, scale, img_shape):
    """Project (K, h, w) crop-space heatmaps onto the (H, W) image.

    Uses the same mapping MMPose applies to keypoints:
        x_img = u * scale_w / w + center_x - scale_w / 2
    """
    hm = np.asarray(heatmaps, dtype=np.float32)
    k, h, w = hm.shape
    sx, sy = scale[0] / w, scale[1] / h
    mat = np.array([[sx, 0, center[0] - scale[0] / 2],
                    [0, sy, center[1] - scale[1] / 2]], dtype=np.float32)
    H, W = img_shape[:2]
    # warpAffine handles at most 4 channels reliably, so warp joint by joint.
    return np.stack([cv2.warpAffine(hm[j], mat, (W, H), flags=cv2.INTER_LINEAR,
                                    borderValue=0) for j in range(k)])


# -------------------------------------------------------------------- metrics
def heatmap_metrics(heatmaps, input_scale, suppress_radius=2, rel_floor=0.1):
    """Per-frame, per-joint quality metrics.

    heatmaps: (T, K, h, w); input_scale: (T, 2) crop size in image px.
    Returns dict of (T, K) arrays: peak, spread_px, second_peak.
    """
    hm = np.asarray(heatmaps, dtype=np.float32)
    T, K, h, w = hm.shape
    flat = hm.reshape(T, K, -1)
    peak = flat.max(-1)
    arg = flat.argmax(-1)
    py, px = np.unravel_index(arg, (h, w))

    ys, xs = np.mgrid[0:h, 0:w]
    # Spread: weighted std of the map after dropping the low background floor.
    wts = np.clip(hm - rel_floor * peak[..., None, None], 0, None)
    tot = wts.sum((-1, -2)) + 1e-12
    mx = (wts * xs).sum((-1, -2)) / tot
    my = (wts * ys).sum((-1, -2)) / tot
    var_x = (wts * (xs - mx[..., None, None]) ** 2).sum((-1, -2)) / tot
    var_y = (wts * (ys - my[..., None, None]) ** 2).sum((-1, -2)) / tot
    px_per_hm = (np.asarray(input_scale)[:, 0] / w + np.asarray(input_scale)[:, 1] / h) / 2
    spread_px = np.sqrt(var_x + var_y) * px_per_hm[:, None]

    # Second peak: highest *local maximum* away from the main peak (a plain
    # "max outside a disc" would just pick up the flank of the main blob).
    padded = np.pad(hm, ((0, 0), (0, 0), (1, 1), (1, 1)), constant_values=-np.inf)
    neigh = np.max([padded[..., 1 + dy:1 + dy + h, 1 + dx:1 + dx + w]
                    for dy in (-1, 0, 1) for dx in (-1, 0, 1) if dy or dx], axis=0)
    is_peak = hm >= neigh
    dist2 = (xs[None, None] - px[..., None, None]) ** 2 + (ys[None, None] - py[..., None, None]) ** 2
    masked = np.where(is_peak & (dist2 > suppress_radius ** 2), hm, 0.0)
    second = masked.reshape(T, K, -1).max(-1)
    second_peak = np.clip(second, 0, None) / np.maximum(peak, 1e-6)

    return dict(peak=peak, spread_px=spread_px, second_peak=second_peak)


def flag_unreliable(metrics, bbox_size, conf_thr=0.3, spread_frac=0.08, second_thr=0.5):
    """Boolean (T, K) mask of predictions we should not trust, plus the reason.

    A joint is unreliable when its heatmap peak is low, the blob is wide
    relative to the person size, or a competing peak is nearly as strong.
    """
    bbox_size = np.asarray(bbox_size, dtype=float)[:, None]
    low_conf = metrics["peak"] < conf_thr
    diffuse = metrics["spread_px"] > spread_frac * bbox_size
    ambiguous = metrics["second_peak"] > second_thr
    return dict(unreliable=low_conf | diffuse | ambiguous,
                low_conf=low_conf, diffuse=diffuse, ambiguous=ambiguous)


def bbox_sizes(bboxes):
    b = np.asarray(bboxes, dtype=float)
    return np.maximum(b[:, 2] - b[:, 0], b[:, 3] - b[:, 1])


def metrics_table(results, **flag_kwargs):
    """Tidy DataFrame (frame, joint, score, peak, spread_px, second_peak, flags...)."""
    m = heatmap_metrics(results["heatmaps"], results["input_scale"])
    flags = flag_unreliable(m, bbox_sizes(results["bboxes"]), **flag_kwargs)
    T, K = m["peak"].shape
    fps = results["meta"]["fps"]
    frames = np.repeat(results["frames"], K)
    df = pd.DataFrame(dict(
        frame=frames, time_s=frames / fps, joint=np.tile(COCO_KEYPOINTS, T),
        x=results["keypoints"][..., 0].ravel(), y=results["keypoints"][..., 1].ravel(),
        score=results["scores"].ravel(),
        **{k: v.ravel() for k, v in m.items()},
        **{k: v.ravel() for k, v in flags.items()},
    ))
    return df


def joint_summary(metrics_df):
    """Per-joint averages and % of frames flagged unreliable."""
    g = metrics_df.groupby("joint", sort=False)
    out = pd.DataFrame(dict(
        mean_score=g["score"].mean(), min_score=g["score"].min(),
        mean_spread_px=g["spread_px"].mean(), mean_second_peak=g["second_peak"].mean(),
        pct_unreliable=100 * g["unreliable"].mean(), pct_low_conf=100 * g["low_conf"].mean(),
        pct_diffuse=100 * g["diffuse"].mean(), pct_ambiguous=100 * g["ambiguous"].mean(),
    ))
    return out.reindex(COCO_KEYPOINTS)


# -------------------------------------------------------------- OpenCV drawing
def colorize(hm_img, vmax=1.0):
    """Single (H, W) map -> BGR JET image."""
    norm = np.clip(hm_img / vmax, 0, 1)
    return cv2.applyColorMap((norm * 255).astype(np.uint8), cv2.COLORMAP_JET)


def overlay_heatmap(frame, hm_img, alpha=0.5, vmax=1.0):
    color = colorize(hm_img, vmax)
    # Blend only where there is heat so the background stays visible.
    weight = (np.clip(hm_img / vmax, 0, 1) ** 0.5 * alpha)[..., None]
    return (frame * (1 - weight) + color * weight).astype(np.uint8)


def draw_pose(frame, keypoints, scores, thr=0.3, unreliable=None, radius=4, thickness=2):
    """Skeleton on top of the frame. Low-score joints are skipped; joints
    flagged unreliable are drawn with a red ring."""
    img = frame.copy()
    for a, b in SKELETON_IDX:
        if scores[a] >= thr and scores[b] >= thr:
            pa, pb = tuple(np.round(keypoints[a]).astype(int)), tuple(np.round(keypoints[b]).astype(int))
            cv2.line(img, pa, pb, kpt_color(COCO_KEYPOINTS[b]), thickness, cv2.LINE_AA)
    for j, (x, y) in enumerate(keypoints):
        if scores[j] < thr:
            continue
        c = (int(round(x)), int(round(y)))
        cv2.circle(img, c, radius, kpt_color(COCO_KEYPOINTS[j]), -1, cv2.LINE_AA)
        if unreliable is not None and unreliable[j]:
            cv2.circle(img, c, radius + 4, (0, 0, 255), 2, cv2.LINE_AA)
    return img


def render_frame(results, t, frame, joint=None, alpha=0.55, flags=None):
    """Overlay heatmap (all joints = max over joints, or a single joint) + skeleton."""
    hm_img = heatmap_to_image(results["heatmaps"][t], results["input_center"][t],
                              results["input_scale"][t], frame.shape)
    hm = hm_img[COCO_KEYPOINTS.index(joint)] if joint else hm_img.max(0)
    img = overlay_heatmap(frame, hm, alpha)
    unrel = flags["unreliable"][t] if flags is not None else None
    img = draw_pose(img, results["keypoints"][t], results["scores"][t], unreliable=unrel)
    label = f"frame {int(results['frames'][t])}" + (f" | {joint}" if joint else "")
    cv2.putText(img, label, (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
    return img


def render_heatmap_video(results, out_path, joint=None, alpha=0.55, flags=None, max_width=960):
    """Write an mp4 with the heatmap overlay + skeleton for every processed frame."""
    source = results["meta"]["source"]
    fps = results["meta"]["fps"] / results["meta"].get("stride", 1)
    writer = None
    for t, f_idx in enumerate(results["frames"]):
        frame = read_frame(source, int(f_idx))
        img = render_frame(results, t, frame, joint, alpha, flags)
        if img.shape[1] > max_width:
            s = max_width / img.shape[1]
            img = cv2.resize(img, (max_width, int(img.shape[0] * s)))
        if writer is None:
            writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps,
                                     (img.shape[1], img.shape[0]))
        writer.write(img)
    if writer is not None:
        writer.release()
    return out_path


# --------------------------------------------------------- Matplotlib figures
def plot_joint_heatmap_grid(results, t, frame, out_path=None, crop=True):
    """17 panels: each joint's heatmap over the frame, titled with its metrics."""
    hm_img = heatmap_to_image(results["heatmaps"][t], results["input_center"][t],
                              results["input_scale"][t], frame.shape)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    m = heatmap_metrics(results["heatmaps"][t:t + 1], results["input_scale"][t:t + 1])
    if crop:
        x1, y1, x2, y2 = results["bboxes"][t]
        pad = 0.1 * max(x2 - x1, y2 - y1)
        H, W = frame.shape[:2]
        x1, y1 = int(max(0, x1 - pad)), int(max(0, y1 - pad))
        x2, y2 = int(min(W, x2 + pad)), int(min(H, y2 + pad))
    else:
        x1, y1, (y2, x2) = 0, 0, frame.shape[:2]

    fig, axes = plt.subplots(3, 6, figsize=(18, 10))
    for j, ax in enumerate(axes.ravel()):
        ax.axis("off")
        if j == len(COCO_KEYPOINTS):
            ax.imshow(rgb[y1:y2, x1:x2])
            ax.imshow(hm_img.max(0)[y1:y2, x1:x2], cmap="jet", alpha=0.5, vmin=0, vmax=1)
            ax.set_title("all joints (max)", fontsize=10)
            continue
        if j > len(COCO_KEYPOINTS):
            continue
        ax.imshow(rgb[y1:y2, x1:x2])
        ax.imshow(hm_img[j][y1:y2, x1:x2], cmap="jet", alpha=0.55, vmin=0, vmax=1)
        kx, ky = results["keypoints"][t, j]
        ax.plot(kx - x1, ky - y1, "w+", ms=10, mew=2)
        ax.set_title(f"{COCO_KEYPOINTS[j]}\nconf {m['peak'][0, j]:.2f} | "
                     f"spread {m['spread_px'][0, j]:.0f}px | 2nd {m['second_peak'][0, j]:.2f}",
                     fontsize=8)
    fig.suptitle(f"Joint-confidence heatmaps, frame {int(results['frames'][t])}", fontsize=14)
    fig.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=120)
    return fig


def plot_confidence_timeline(metrics_df, joints=None, conf_thr=0.3, out_path=None):
    """Heatmap peak (confidence) over time per joint, unreliable frames marked."""
    joints = joints or COCO_KEYPOINTS[5:]  # body joints; face points are rarely of interest
    fig, axes = plt.subplots(len(joints), 1, figsize=(12, 1.3 * len(joints)), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, joint in zip(axes, joints):
        d = metrics_df[metrics_df.joint == joint]
        ax.plot(d.time_s, d.score, lw=1.2, color="#2b6cb0")
        bad = d[d.unreliable]
        ax.scatter(bad.time_s, bad.score, s=10, color="#e53e3e", zorder=3)
        ax.axhline(conf_thr, color="gray", ls="--", lw=0.8)
        ax.set_ylim(0, 1.05)
        ax.set_ylabel(joint.replace("_", "\n"), rotation=0, ha="right", va="center", fontsize=8)
        ax.tick_params(labelsize=7)
    axes[-1].set_xlabel("time (s)")
    fig.suptitle("Heatmap confidence per joint (red = flagged unreliable)")
    fig.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=120)
    return fig


def plot_joint_summary(summary, out_path=None):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
    y = np.arange(len(summary))
    a1.barh(y, summary.mean_score, color="#2b6cb0")
    a1.set_yticks(y, summary.index, fontsize=8)
    a1.invert_yaxis()
    a1.set_xlim(0, 1)
    a1.set_xlabel("mean heatmap confidence")
    left = 0
    for col, c in (("pct_low_conf", "#e53e3e"), ("pct_diffuse", "#dd6b20"), ("pct_ambiguous", "#805ad5")):
        a2.barh(y, summary[col], left=left, color=c, label=col.replace("pct_", ""))
        left = left + summary[col]
    a2.set_yticks(y, summary.index, fontsize=8)
    a2.invert_yaxis()
    a2.set_xlabel("% of frames (reasons can overlap)")
    a2.legend(fontsize=8)
    a2.set_title("Why predictions were flagged unreliable")
    fig.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=120)
    return fig


def plot_metric_scatter(metrics_df, out_path=None):
    """Confidence vs spread: wide, low-peak heatmaps cluster bottom-right."""
    fig, ax = plt.subplots(figsize=(7, 5))
    body = metrics_df[~metrics_df.joint.isin(COCO_KEYPOINTS[:5])]
    sc = ax.scatter(body.spread_px, body.score, c=body.second_peak, cmap="viridis",
                    s=6, alpha=0.6, vmin=0, vmax=1)
    fig.colorbar(sc, label="second-peak ratio")
    ax.set_xlabel("heatmap spread (px)")
    ax.set_ylabel("heatmap peak (confidence)")
    ax.set_title("Heatmap shape vs confidence (body joints)")
    fig.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=120)
    return fig

In [ ]:
%%writefile posestudy/kinovea.py
"""Validate predicted keypoints / joint angles against Kinovea measurements.

Kinovea (https://www.kinovea.org) is used to manually track markers / joints
and measure angles frame by frame. Its spreadsheet export differs between
versions and languages, so loading is driven by a small JSON mapping that
says which column holds what. Example `mapping.json`:

    {
      "time_column": "Time (ms)",          # or "frame_column": "Frame"
      "time_unit": "ms",                   # ms | s
      "points": {
        "left_knee":  {"x": "Knee/X",  "y": "Knee/Y"},
        "left_ankle": {"x": "Ankle/X", "y": "Ankle/Y"}
      },
      "angles": {"left_knee": "Knee angle"},
      "coords": {"px_per_unit": 1.0, "origin_px": [0, 0], "y_up": false}
    }

`coords` converts Kinovea coordinates to image pixels:
    x_px = origin_x + x * px_per_unit
    y_px = origin_y - y * px_per_unit  (if y_up)   else  origin_y + y * px_per_unit
Uncalibrated Kinovea data in pixels with a top-left origin needs the
defaults. If the coordinate system was calibrated (metres/cm, moved origin),
either fill these in or validate with joint angles, which do not depend on
the coordinate system.
"""

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from .skeleton import COCO_KEYPOINTS, compute_joint_angles


# --------------------------------------------------------------------- loading
def read_table(path):
    """Read a Kinovea CSV/TSV/XLSX, coping with ';' separators and ',' decimals."""
    if str(path).lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    with open(path, encoding="utf-8-sig", errors="replace") as f:
        head = f.read(4096)
    header = head.splitlines()[0] if head else ""
    # Decide from the header row: data rows may use ',' as the decimal mark.
    sep = max((";", "\t", ","), key=header.count) if header else ","
    decimal = "," if sep in (";", "\t") and _has_comma_decimals(head, sep) else "."
    df = pd.read_csv(path, sep=sep, decimal=decimal, encoding="utf-8-sig")
    df.columns = [str(c).strip() for c in df.columns]
    return df


def _has_comma_decimals(text, sep):
    cells = [c for line in text.splitlines()[1:] for c in line.split(sep)]
    return any(c.strip().replace(",", "", 1).lstrip("-").isdigit() and "," in c for c in cells)


def inspect(path, n=5):
    """Print the columns and first rows, to help write mapping.json."""
    df = read_table(path)
    print(f"{path}: {len(df)} rows")
    print("columns:", list(df.columns))
    print(df.head(n).to_string())
    return df


def load_kinovea(path, mapping, fps, frame_offset=0):
    """Return (points_df, angles_df) in tidy form, keyed by video frame.

    points_df: frame, joint, x, y   (image pixels)
    angles_df: frame, angle, value  (degrees)
    """
    if isinstance(mapping, str):
        with open(mapping) as f:
            mapping = json.load(f)
    df = read_table(path)
    frame = _frame_index(df, mapping, fps) + frame_offset

    coords = mapping.get("coords", {})
    ppu = float(coords.get("px_per_unit", 1.0))
    ox, oy = coords.get("origin_px", [0, 0])
    y_up = bool(coords.get("y_up", False))

    pts = []
    for joint, cols in mapping.get("points", {}).items():
        if joint not in COCO_KEYPOINTS:
            raise ValueError(f"'{joint}' is not a COCO keypoint name: {COCO_KEYPOINTS}")
        x = pd.to_numeric(df[cols["x"]], errors="coerce").to_numpy() * ppu + ox
        y = pd.to_numeric(df[cols["y"]], errors="coerce").to_numpy() * ppu
        y = oy - y if y_up else oy + y
        pts.append(pd.DataFrame(dict(frame=frame, joint=joint, x=x, y=y)))
    ang = []
    for name, col in mapping.get("angles", {}).items():
        v = pd.to_numeric(df[col], errors="coerce").to_numpy()
        ang.append(pd.DataFrame(dict(frame=frame, angle=name, value=v)))

    points = pd.concat(pts).dropna() if pts else pd.DataFrame(columns=["frame", "joint", "x", "y"])
    angles = pd.concat(ang).dropna() if ang else pd.DataFrame(columns=["frame", "angle", "value"])
    points["frame"] = points.frame.astype(int)
    angles["frame"] = angles.frame.astype(int)
    # Several Kinovea rows can round to the same frame; average them.
    points = points.groupby(["frame", "joint"], as_index=False)[["x", "y"]].mean()
    angles = angles.groupby(["frame", "angle"], as_index=False)["value"].mean()
    return points, angles


def _frame_index(df, mapping, fps):
    if "frame_column" in mapping:
        return pd.to_numeric(df[mapping["frame_column"]], errors="coerce").round().to_numpy(dtype=float)
    col = mapping.get("time_column") or df.columns[0]
    t = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=float)
    t_s = t / 1000.0 if mapping.get("time_unit", "ms") == "ms" else t
    return np.round(t_s * fps)  # float so missing times stay NaN


# ------------------------------------------------------------------ alignment
def fit_similarity(src, dst):
    """Least-squares scale + translation (+ optional y flip) mapping src -> dst.

    Only use this when Kinovea's coordinate system is unknown: it removes any
    global offset/scale between the two, so a global bias in the model would
    be hidden too. Returns (params, transform_fn).
    """
    src, dst = np.asarray(src, float), np.asarray(dst, float)
    best = None
    for flip in (1.0, -1.0):
        s2 = src * [1.0, flip]
        sc, dc = s2.mean(0), dst.mean(0)
        a, b = s2 - sc, dst - dc
        scale = (a * b).sum() / max((a * a).sum(), 1e-12)
        resid = np.linalg.norm(b - scale * a, axis=1).mean()
        if best is None or resid < best[0]:
            best = (resid, flip, scale, dc - scale * sc)
    _, flip, scale, t = best

    def transform(p):
        return np.asarray(p, float) * [1.0, flip] * scale + t
    return dict(scale=float(scale), y_flip=flip < 0, translation=t.tolist()), transform


# ------------------------------------------------------------------ comparison
def predictions_table(results, metrics_df=None):
    """Tidy predicted keypoints (frame, joint, x_pred, y_pred, score, bbox_size[, unreliable])."""
    b = results["bboxes"]
    size = np.maximum(b[:, 2] - b[:, 0], b[:, 3] - b[:, 1])
    T, K = results["scores"].shape
    df = pd.DataFrame(dict(
        frame=np.repeat(results["frames"], K), joint=np.tile(COCO_KEYPOINTS, T),
        x_pred=results["keypoints"][..., 0].ravel(), y_pred=results["keypoints"][..., 1].ravel(),
        score=results["scores"].ravel(), bbox_size=np.repeat(size, K)))
    if metrics_df is not None:
        df = df.merge(metrics_df[["frame", "joint", "spread_px", "second_peak", "unreliable"]],
                      on=["frame", "joint"], how="left")
    return df


def predicted_angles(results):
    ang = compute_joint_angles(results["keypoints"])
    return pd.DataFrame([dict(frame=int(f), angle=name, pred=float(v[i]))
                         for name, v in ang.items() for i, f in enumerate(results["frames"])])


def match_keypoints(pred_df, gt_points):
    m = pred_df.merge(gt_points.rename(columns={"x": "x_gt", "y": "y_gt"}), on=["frame", "joint"])
    m["error_px"] = np.hypot(m.x_pred - m.x_gt, m.y_pred - m.y_gt)
    m["error_norm"] = m.error_px / m.bbox_size
    return m


def best_frame_offset(pred_df, gt_points, max_offset=10):
    """Shift Kinovea frames by -max..+max and keep the shift with the lowest
    median error; fixes a start-time mismatch between the two."""
    scores = {}
    for off in range(-max_offset, max_offset + 1):
        g = gt_points.assign(frame=gt_points.frame + off)
        m = match_keypoints(pred_df, g)
        if len(m):
            scores[off] = m.error_px.median()
    return min(scores, key=scores.get), scores


def keypoint_summary(matched, pck_alphas=(0.05, 0.1)):
    """Per-joint error statistics. PCK@a = % of frames with error < a * person box size."""
    def stats(d):
        out = dict(n=len(d), mean_err_px=d.error_px.mean(), median_err_px=d.error_px.median(),
                   rmse_px=np.sqrt((d.error_px ** 2).mean()), max_err_px=d.error_px.max(),
                   mean_score=d.score.mean())
        for a in pck_alphas:
            out[f"PCK@{a}"] = 100 * (d.error_norm < a).mean()
        return pd.Series(out)
    per_joint = matched.groupby("joint", sort=False).apply(stats, include_groups=False)
    overall = stats(matched).rename("ALL")
    return pd.concat([per_joint, overall.to_frame().T])


def confidence_vs_error(matched, conf_bins=(0, 0.3, 0.5, 0.7, 0.9, 1.01)):
    """Does a low heatmap confidence actually mean a larger error?"""
    m = matched.copy()
    m["conf_bin"] = pd.cut(m.score, conf_bins, right=False)
    t = m.groupby("conf_bin", observed=True).error_px.agg(["count", "mean", "median"])
    rho = m[["score", "error_px"]].corr(method="spearman").iloc[0, 1]
    if "unreliable" in m:
        flagged = m.groupby("unreliable").error_px.agg(["count", "mean", "median"])
    else:
        flagged = None
    return t, rho, flagged


def angle_agreement(pred_ang, gt_angles):
    """MAE, RMSE, bias, Bland-Altman 95% limits and Pearson r per angle."""
    m = pred_ang.merge(gt_angles.rename(columns={"value": "kinovea"}), on=["frame", "angle"]).dropna()
    rows = {}
    for name, d in m.groupby("angle"):
        diff = d.pred - d.kinovea
        sd = diff.std(ddof=1) if len(d) > 1 else np.nan
        rows[name] = dict(n=len(d), MAE_deg=diff.abs().mean(), RMSE_deg=np.sqrt((diff ** 2).mean()),
                          bias_deg=diff.mean(), LoA_low=diff.mean() - 1.96 * sd,
                          LoA_high=diff.mean() + 1.96 * sd,
                          pearson_r=d.pred.corr(d.kinovea) if len(d) > 2 else np.nan)
    return m, pd.DataFrame(rows).T


# ----------------------------------------------------------------------- plots
def plot_error_per_joint(summary, out_path=None):
    s = summary.drop(index="ALL", errors="ignore")
    fig, ax = plt.subplots(figsize=(9, 0.45 * len(s) + 1.5))
    y = np.arange(len(s))
    ax.barh(y, s.mean_err_px, color="#2b6cb0", label="mean")
    ax.scatter(s.median_err_px, y, color="k", marker="|", s=200, label="median", zorder=3)
    ax.set_yticks(y, s.index)
    ax.invert_yaxis()
    ax.set_xlabel("error vs Kinovea (px)")
    ax.legend()
    ax.set_title("Keypoint error per joint")
    fig.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=120)
    return fig


def plot_trajectories(matched, joint, fps, out_path=None):
    d = matched[matched.joint == joint].sort_values("frame")
    t = d.frame / fps
    fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
    for ax, c in zip(axes[:2], ("x", "y")):
        ax.plot(t, d[f"{c}_gt"], "k-", lw=1.5, label="Kinovea")
        ax.plot(t, d[f"{c}_pred"], "-", color="#dd6b20", lw=1.2, label="MMPose")
        ax.set_ylabel(f"{c} (px)")
        ax.legend(fontsize=8)
    axes[2].plot(t, d.error_px, color="#e53e3e", lw=1)
    if "unreliable" in d:
        bad = d[d.unreliable.fillna(False).astype(bool)]
        axes[2].scatter(bad.frame / fps, bad.error_px, s=12, color="#805ad5", label="flagged unreliable", zorder=3)
        axes[2].legend(fontsize=8)
    axes[2].set_ylabel("error (px)")
    axes[2].set_xlabel("time (s)")
    fig.suptitle(f"{joint}: MMPose vs Kinovea")
    fig.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=120)
    return fig


def plot_confidence_vs_error(matched, out_path=None):
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(matched.score, matched.error_px, s=8, alpha=0.5, color="#2b6cb0")
    ax.set_xlabel("heatmap confidence")
    ax.set_ylabel("error vs Kinovea (px)")
    ax.set_title("Is low heatmap confidence a warning sign?")
    fig.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=120)
    return fig


def plot_angles(angle_matched, fps, out_path=None):
    names = sorted(angle_matched.angle.unique())
    fig, axes = plt.subplots(len(names), 2, figsize=(13, 3.2 * len(names)), squeeze=False)
    for (a1, a2), name in zip(axes, names):
        d = angle_matched[angle_matched.angle == name].sort_values("frame")
        a1.plot(d.frame / fps, d.kinovea, "k-", label="Kinovea")
        a1.plot(d.frame / fps, d.pred, "-", color="#dd6b20", label="MMPose")
        a1.set_title(f"{name} angle")
        a1.set_xlabel("time (s)")
        a1.set_ylabel("deg")
        a1.legend(fontsize=8)
        # Bland-Altman
        mean, diff = (d.pred + d.kinovea) / 2, d.pred - d.kinovea
        bias, sd = diff.mean(), diff.std(ddof=1)
        a2.scatter(mean, diff, s=10, alpha=0.6)
        for v, ls in ((bias, "-"), (bias + 1.96 * sd, "--"), (bias - 1.96 * sd, "--")):
            a2.axhline(v, color="gray", ls=ls)
        a2.set_title(f"Bland-Altman: bias {bias:.1f}°, LoA ±{1.96 * sd:.1f}°")
        a2.set_xlabel("mean of methods (deg)")
        a2.set_ylabel("MMPose - Kinovea (deg)")
    fig.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=120)
    return fig

In [ ]:
%%writefile scripts/run_pose.py
"""Step 1 - run MMPose heatmap pose estimation on a video / image / image folder.

    python scripts/run_pose.py --input data/trial.mp4 --out results/trial
    python scripts/run_pose.py --input data/trial.mp4 --out results/trial --model resnet50 --no-detector
"""

import argparse
import os.path as osp
import sys

sys.path.insert(0, osp.dirname(osp.dirname(osp.abspath(__file__))))

from posestudy.estimator import MODEL_ZOO, PersonDetector, load_pose_model, run_on_video  # noqa: E402


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--input", required=True, help="video file, image, or folder of images")
    p.add_argument("--out", required=True, help="output folder")
    p.add_argument("--model", default="hrnet-w32", choices=sorted(MODEL_ZOO))
    p.add_argument("--config", help="custom MMPose top-down heatmap config (overrides --model)")
    p.add_argument("--checkpoint", help="custom checkpoint path/URL")
    p.add_argument("--device", default=None, help="cuda:0 / cpu (default: auto)")
    p.add_argument("--no-detector", action="store_true",
                   help="use the whole frame as the person box (subject fills the frame)")
    p.add_argument("--stride", type=int, default=1, help="process every n-th frame")
    p.add_argument("--max-frames", type=int, default=None)
    p.add_argument("--fps", type=float, default=None, help="override video fps")
    args = p.parse_args()

    model = load_pose_model(args.model, args.device, args.config, args.checkpoint)
    detector = None if args.no_detector else PersonDetector(args.device)
    run_on_video(model, args.input, args.out, detector=detector, stride=args.stride,
                 max_frames=args.max_frames, fps=args.fps,
                 model_name=args.config or args.model)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/analyse_heatmaps.py
"""Step 2 - visualise and analyse the joint-confidence heatmaps.

    python scripts/analyse_heatmaps.py --results results/trial
    python scripts/analyse_heatmaps.py --results results/trial --video --joint left_knee

Writes into <results>/heatmap_analysis/:
    heatmap_metrics.csv       per frame x joint: score, spread, second peak, flags
    joint_summary.csv         per joint averages + % unreliable frames
    confidence_timeline.png   confidence over time, unreliable frames in red
    joint_summary.png         mean confidence and reasons for flags per joint
    confidence_vs_spread.png  heatmap shape vs confidence
    grid_frame_XXXX.png       17-panel per-joint heatmaps (best/worst/middle frame)
    overlay_frame_XXXX.jpg    OpenCV overlay of all heatmaps + skeleton
    heatmap_overlay.mp4       (with --video) overlay for every processed frame
"""

import argparse
import os
import os.path as osp
import sys

sys.path.insert(0, osp.dirname(osp.dirname(osp.abspath(__file__))))

import cv2  # noqa: E402
import matplotlib  # noqa: E402

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402

from posestudy import heatmaps as hm  # noqa: E402


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--results", required=True, help="folder written by run_pose.py")
    p.add_argument("--conf-thr", type=float, default=0.3, help="flag joints with peak below this")
    p.add_argument("--spread-frac", type=float, default=0.08,
                   help="flag joints whose heatmap spread > this fraction of the person box")
    p.add_argument("--second-thr", type=float, default=0.5,
                   help="flag joints whose second peak is > this fraction of the main peak")
    p.add_argument("--frames", type=int, nargs="*", help="frame indices to plot (default: auto)")
    p.add_argument("--video", action="store_true", help="also render an overlay video")
    p.add_argument("--joint", default=None, help="overlay video shows only this joint")
    args = p.parse_args()

    res = hm.load_results(args.results)
    out = osp.join(args.results, "heatmap_analysis")
    os.makedirs(out, exist_ok=True)
    flag_kw = dict(conf_thr=args.conf_thr, spread_frac=args.spread_frac, second_thr=args.second_thr)

    metrics = hm.metrics_table(res, **flag_kw)
    metrics.to_csv(osp.join(out, "heatmap_metrics.csv"), index=False)
    summary = hm.joint_summary(metrics)
    summary.to_csv(osp.join(out, "joint_summary.csv"))
    print(summary.round(3).to_string())

    hm.plot_confidence_timeline(metrics, conf_thr=args.conf_thr,
                                out_path=osp.join(out, "confidence_timeline.png"))
    hm.plot_joint_summary(summary, out_path=osp.join(out, "joint_summary.png"))
    hm.plot_metric_scatter(metrics, out_path=osp.join(out, "confidence_vs_spread.png"))
    plt.close("all")

    # Frames to inspect: best, median and worst mean body-joint confidence.
    body_conf = res["scores"][:, 5:].mean(1)
    if args.frames:
        ts = [int(np.where(res["frames"] == f)[0][0]) for f in args.frames]
    else:
        order = np.argsort(body_conf)
        ts = sorted({int(order[-1]), int(order[len(order) // 2]), int(order[0])})
    m = hm.heatmap_metrics(res["heatmaps"], res["input_scale"])
    flags = hm.flag_unreliable(m, hm.bbox_sizes(res["bboxes"]), **flag_kw)
    for t in ts:
        f_idx = int(res["frames"][t])
        frame = hm.read_frame(res["meta"]["source"], f_idx)
        hm.plot_joint_heatmap_grid(res, t, frame, out_path=osp.join(out, f"grid_frame_{f_idx:05d}.png"))
        cv2.imwrite(osp.join(out, f"overlay_frame_{f_idx:05d}.jpg"), hm.render_frame(res, t, frame, flags=flags))
        plt.close("all")
        print(f"frame {f_idx}: mean body confidence {body_conf[t]:.3f}")

    if args.video:
        path = hm.render_heatmap_video(res, osp.join(out, "heatmap_overlay.mp4"), joint=args.joint, flags=flags)
        print("video:", path)
    print("Saved analysis to", out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/compare_kinovea.py
"""Step 3 - validate predicted keypoints / joint angles against Kinovea.

    # look at the export first, then write a mapping.json (see data/kinovea_mapping_example.json)
    python scripts/compare_kinovea.py --kinovea data/trial_kinovea.csv --inspect

    python scripts/compare_kinovea.py --results results/trial \
        --kinovea data/trial_kinovea.csv --mapping data/kinovea_mapping.json --auto-offset 10

Writes into <results>/kinovea_validation/:
    matched_keypoints.csv, keypoint_summary.csv (error px, RMSE, PCK per joint)
    confidence_vs_error.csv, angle_agreement.csv (MAE, RMSE, bias, Bland-Altman LoA, r)
    error_per_joint.png, trajectory_<joint>.png, confidence_vs_error.png, angles.png
"""

import argparse
import json
import os
import os.path as osp
import sys

sys.path.insert(0, osp.dirname(osp.dirname(osp.abspath(__file__))))

import matplotlib  # noqa: E402

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

from posestudy import heatmaps as hm  # noqa: E402
from posestudy import kinovea as kv  # noqa: E402


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--kinovea", required=True, help="Kinovea export (.csv / .txt / .xlsx)")
    p.add_argument("--inspect", action="store_true", help="only print the export's columns")
    p.add_argument("--results", help="folder written by run_pose.py")
    p.add_argument("--mapping", help="JSON column mapping (see posestudy/kinovea.py)")
    p.add_argument("--frame-offset", type=int, default=0, help="add to Kinovea frame numbers")
    p.add_argument("--auto-offset", type=int, default=0,
                   help="search +-N frames for the best time alignment")
    p.add_argument("--auto-align", action="store_true",
                   help="fit scale/translation/y-flip from Kinovea coords to pixels "
                        "(only if the Kinovea coordinate system is unknown)")
    args = p.parse_args()

    if args.inspect:
        kv.inspect(args.kinovea)
        return
    if not (args.results and args.mapping):
        p.error("--results and --mapping are required (or use --inspect)")

    res = hm.load_results(args.results)
    fps = res["meta"]["fps"]
    out = osp.join(args.results, "kinovea_validation")
    os.makedirs(out, exist_ok=True)
    gt_pts, gt_ang = kv.load_kinovea(args.kinovea, args.mapping, fps, args.frame_offset)
    print(f"Kinovea: {len(gt_pts)} point samples ({gt_pts.joint.nunique()} joints), "
          f"{len(gt_ang)} angle samples")

    metrics = hm.metrics_table(res)
    pred = kv.predictions_table(res, metrics)
    report = dict(frame_offset=args.frame_offset)

    if len(gt_pts):
        if args.auto_offset:
            off, _ = kv.best_frame_offset(pred, gt_pts, args.auto_offset)
            print(f"best extra frame offset: {off:+d}")
            gt_pts = gt_pts.assign(frame=gt_pts.frame + off)
            gt_ang = gt_ang.assign(frame=gt_ang.frame + off)
            report["auto_offset"] = off
        if args.auto_align:
            m0 = kv.match_keypoints(pred, gt_pts)
            params, f = kv.fit_similarity(m0[["x_gt", "y_gt"]].to_numpy(), m0[["x_pred", "y_pred"]].to_numpy())
            gt_pts[["x", "y"]] = f(gt_pts[["x", "y"]].to_numpy())
            print("auto-align:", params)
            report["auto_align"] = params

        matched = kv.match_keypoints(pred, gt_pts)
        if matched.empty:
            sys.exit("No overlapping frames/joints - check the time column, fps and frame offset.")
        matched.to_csv(osp.join(out, "matched_keypoints.csv"), index=False)
        summary = kv.keypoint_summary(matched)
        summary.to_csv(osp.join(out, "keypoint_summary.csv"))
        print("\nKeypoint error vs Kinovea:\n", summary.round(2).to_string())

        bins, rho, flagged = kv.confidence_vs_error(matched)
        bins.to_csv(osp.join(out, "confidence_vs_error.csv"))
        print(f"\nError by heatmap confidence (Spearman rho score~error = {rho:.2f}):\n{bins.round(2)}")
        if flagged is not None:
            print("\nError for frames flagged unreliable from heatmaps:\n", flagged.round(2))
        report["spearman_conf_error"] = rho

        kv.plot_error_per_joint(summary, osp.join(out, "error_per_joint.png"))
        kv.plot_confidence_vs_error(matched, osp.join(out, "confidence_vs_error.png"))
        for joint in matched.joint.unique():
            kv.plot_trajectories(matched, joint, fps, osp.join(out, f"trajectory_{joint}.png"))
        plt.close("all")

    if len(gt_ang):
        ang_m, ang_s = kv.angle_agreement(kv.predicted_angles(res), gt_ang)
        if ang_m.empty:
            print("No overlapping angle frames.")
        else:
            ang_m.to_csv(osp.join(out, "matched_angles.csv"), index=False)
            ang_s.to_csv(osp.join(out, "angle_agreement.csv"))
            print("\nJoint-angle agreement vs Kinovea:\n", ang_s.round(2).to_string())
            kv.plot_angles(ang_m, fps, osp.join(out, "angles.png"))
            plt.close("all")

    with open(osp.join(out, "report.json"), "w") as f:
        json.dump(report, f, indent=2, default=float)
    print("\nSaved validation to", out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile data/kinovea_mapping_example.json
{
  "time_column": "Time (ms)",
  "time_unit": "ms",
  "points": {
    "left_hip":   {"x": "Hip/X",   "y": "Hip/Y"},
    "left_knee":  {"x": "Knee/X",  "y": "Knee/Y"},
    "left_ankle": {"x": "Ankle/X", "y": "Ankle/Y"}
  },
  "angles": {
    "left_knee": "Knee angle"
  },
  "coords": {"px_per_unit": 1.0, "origin_px": [0, 0], "y_up": false}
}

## 3. Run pose estimation

In [ ]:
import sys, os
sys.path.insert(0, "/content/pose-heatmap-study")
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, Video, display
from posestudy import heatmaps as hm, kinovea as kv
from posestudy.estimator import load_pose_model, PersonDetector, run_on_video

pose_model = load_pose_model("hrnet-w32")   # or "hrnet-w48", "resnet50"
detector = PersonDetector()                 # set to None if the subject fills the frame

In [ ]:
#@title Input: sample image, or upload your own video
SOURCE = "sample_image"  #@param ["sample_image", "upload"]
os.makedirs("data", exist_ok=True)
if SOURCE == "sample_image":
    !wget -q -O data/sample.jpg https://raw.githubusercontent.com/open-mmlab/mmpose/main/tests/data/coco/000000000785.jpg
    INPUT = "data/sample.jpg"
else:
    from google.colab import files
    up = files.upload()
    INPUT = "data/" + list(up)[0]
    os.replace(list(up)[0], INPUT)
print("input:", INPUT)

In [ ]:
RESULTS = "results/" + os.path.splitext(os.path.basename(INPUT))[0]
STRIDE = 1          # process every n-th frame (use 2-3 for long videos)
MAX_FRAMES = None   # e.g. 300 for a quick test
df = run_on_video(pose_model, INPUT, RESULTS, detector=detector, stride=STRIDE, max_frames=MAX_FRAMES)
df.head(17)

## 4. Joint-confidence heatmaps

For every joint the model outputs a 64×48 heatmap; its peak position is the
keypoint and the peak height is the confidence. The heatmap shape is used to
judge localisation quality:

* **peak** – confidence; low → joint probably occluded / out of frame
* **spread (px)** – width of the blob in image pixels; wide → uncertain position
* **second peak** – strongest other local maximum ÷ main peak; high → the model is torn between two places (typical for left/right swaps or limbs crossing)

A joint is flagged **unreliable** when peak < 0.3, spread > 8 % of the person box, or second peak > 0.5.

In [ ]:
res = hm.load_results(RESULTS)
metrics = hm.metrics_table(res)        # conf_thr=0.3, spread_frac=0.08, second_thr=0.5
summary = hm.joint_summary(metrics)
summary.round(3)

In [ ]:
t = int(np.argmin(res["scores"][:, 5:].mean(1)))   # frame with the lowest body-joint confidence
frame = hm.read_frame(res["meta"]["source"], int(res["frames"][t]))
hm.plot_joint_heatmap_grid(res, t, frame); plt.show()

m = hm.heatmap_metrics(res["heatmaps"], res["input_scale"])
flags = hm.flag_unreliable(m, hm.bbox_sizes(res["bboxes"]))
overlay = hm.render_frame(res, t, frame, flags=flags)   # red ring = flagged joint
plt.figure(figsize=(10, 7)); plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()

In [ ]:
if len(res["frames"]) > 1:
    hm.plot_confidence_timeline(metrics); plt.show()
hm.plot_joint_summary(summary); plt.show()
hm.plot_metric_scatter(metrics); plt.show()

In [ ]:
#@title Heatmap overlay video + save all analysis files
VIDEO_FLAG = "--video" if len(res["frames"]) > 1 else ""
!python scripts/analyse_heatmaps.py --results {RESULTS} {VIDEO_FLAG}
if VIDEO_FLAG:
    !ffmpeg -y -loglevel error -i {RESULTS}/heatmap_analysis/heatmap_overlay.mp4 -vcodec libx264 {RESULTS}/heatmap_analysis/heatmap_overlay_h264.mp4
    display(Video(RESULTS + "/heatmap_analysis/heatmap_overlay_h264.mp4", embed=True, width=720))

## 5. Validation against Kinovea

1. Open the **same video** in Kinovea, track the joints you care about
   (trajectory tool on hip / knee / ankle …) and/or add angle tools.
2. Export the data to a spreadsheet / CSV (the menu name differs by Kinovea version).
3. Upload it here, look at its columns, and fill the mapping in the next cell:
   which column is time, which columns are X/Y of which COCO joint, which are angles.

**Coordinates.** If Kinovea was *not* calibrated the values are pixels; set
`origin_px`/`y_up` to match its coordinate system (e.g. origin bottom-left and
y pointing up → `"origin_px": [0, IMAGE_HEIGHT], "y_up": true`). If it *was*
calibrated (metres), fill in `px_per_unit` or rely on **joint angles**, which
are unit-free. `AUTO_ALIGN` can fit the scale/offset automatically, but it also
hides any global bias of the model, so report that you used it.

In [ ]:
from google.colab import files
up = files.upload()
KINOVEA = "data/" + list(up)[0]; os.replace(list(up)[0], KINOVEA)
kin = kv.inspect(KINOVEA)

In [ ]:
H = res["meta"]["height"]
mapping = {
    "time_column": "Time (ms)",   # or "frame_column": "Frame"
    "time_unit": "ms",
    "points": {                   # COCO joint name -> Kinovea X/Y columns
        "left_hip":   {"x": "Hip/X",   "y": "Hip/Y"},
        "left_knee":  {"x": "Knee/X",  "y": "Knee/Y"},
        "left_ankle": {"x": "Ankle/X", "y": "Ankle/Y"},
    },
    "angles": {"left_knee": "Knee angle"},
    "coords": {"px_per_unit": 1.0, "origin_px": [0, 0], "y_up": False},
}
import json; json.dump(mapping, open("data/kinovea_mapping.json", "w"), indent=2)
AUTO_OFFSET = 10    # search +-10 frames for the best time alignment (0 = off)
AUTO_ALIGN = False

In [ ]:
ALIGN_FLAG = "--auto-align" if AUTO_ALIGN else ""
!python scripts/compare_kinovea.py --results {RESULTS} --kinovea {KINOVEA} --mapping data/kinovea_mapping.json --auto-offset {AUTO_OFFSET} {ALIGN_FLAG}
import glob
for png in sorted(glob.glob(RESULTS + "/kinovea_validation/*.png")):
    print(png); display(Image(png, width=900))

## 6. Save results

In [ ]:
from google.colab import files
!zip -qr results.zip results
if USE_DRIVE:
    !cp -r results "{DRIVE_DIR}/"
    print("copied to", DRIVE_DIR)
files.download("results.zip")